In [ ]:
import numpy as np
from mstcvi import treelhouette_score
from sklearn.metrics import silhouette_score

In [2]:
import numpy as np
import quitefastmst

def get_euclidean_mst(X, *, M=0, **mst_euclid_kwargs):
    result = quitefastmst.mst_euclid(X, M=M, **mst_euclid_kwargs)
    mst_dist, mst_index = result[0], result[1]
    return np.asarray(mst_dist), np.asarray(mst_index)

In [3]:
X = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
    [2.0, 2.0],
    [10.0, 5.0],
    [11.0, 7.0],
    [3.0, 12.0],
    [4.0, 12.0],
])
labels = np.array([0, 0, 0, 1, 1, 2, 2])

In [4]:
mst_dist, mst_index = get_euclidean_mst(X)

print(f"\nmst_dist  -- kształt {mst_dist.shape} (n-1 = {X.shape[0]-1} krawędzi)")
print(mst_dist)

print(f"\nmst_index -- kształt {mst_index.shape}")
print(mst_index)


mst_dist  -- kształt (6,) (n-1 = 6 krawędzi)
[1.         1.41421356 2.23606798 2.23606798 8.54400375 8.60232527]

mst_index -- kształt (6, 2)
[[5 6]
 [0 1]
 [0 2]
 [3 4]
 [2 3]
 [4 6]]


In [5]:
labels = np.asarray(labels)
unique_labels, contig_labels = np.unique(labels, return_inverse=True)

u, v = mst_index[:, 0], mst_index[:, 1]
same_cluster = contig_labels[u] == contig_labels[v]

edge_labels = np.full(mst_dist.shape[0], -1, dtype=int)
edge_labels[same_cluster] = contig_labels[u[same_cluster]]

cut_endpoint_labels = np.full((mst_dist.shape[0], 2), -1, dtype=int)
cut_mask = ~same_cluster
cut_endpoint_labels[cut_mask, 0] = contig_labels[u[cut_mask]]
cut_endpoint_labels[cut_mask, 1] = contig_labels[v[cut_mask]]

In [6]:
cut_endpoint_labels

array([[-1, -1],
       [-1, -1],
       [-1, -1],
       [-1, -1],
       [ 0,  1],
       [ 1,  2]])

In [7]:
k = len(np.unique(labels))
k

3

In [8]:
b_per_cluster = np.full(k, np.inf)

cut_mask = cut_endpoint_labels[:, 0] != -1
cut_weights = mst_dist[cut_mask]

for col in (0, 1):
    cluster_ids = cut_endpoint_labels[cut_mask, col]
    np.minimum.at(b_per_cluster, cluster_ids, cut_weights)

In [10]:
t = treelhouette_score(X, labels)
print("Treelhouette score: ", t)

Treelhouette score:  0.7987018152363734


In [12]:
s = silhouette_score(X, labels)
print("Silhouette score: ", s)

Silhouette score:  0.8206991502455787


In [2]:
"""Thin, cached wrapper around clustbench for this project's needs."""

from pathlib import Path
from typing import Iterator, NamedTuple

import clustbench
import numpy as np

# Path(__file__).resolve().parent 
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "clustering-data-v1"

DEFAULT_BATTERIES = ("fcps", "graves", "other", "sipu", "uci", "wut")


class BenchmarkDataset(NamedTuple):
    battery: str
    name: str
    X: np.ndarray
    reference_labels: list[np.ndarray]  # l >= 1 etykietowań referencyjnych


def iter_benchmark_datasets(
    batteries: tuple[str, ...] = DEFAULT_BATTERIES,
) -> Iterator[BenchmarkDataset]:
    """Yield every dataset in the given batteries, preprocessed.

    Preprocessing matches Gagolewski, Bartoszuk & Cena (2021), Sec. 3.2:
    zero-variance columns removed, tiny noise added for uniqueness.
    """
    for battery in batteries:
        for name in clustbench.get_dataset_names(battery, path=DATA_PATH):
            b = clustbench.load_dataset(
                battery, name, path=DATA_PATH, preprocess=True
            )
            labels = b.labels if isinstance(b.labels, list) else [b.labels]
            yield BenchmarkDataset(battery, name, b.data, labels)

In [3]:
print(clustbench.get_battery_names(path=DATA_PATH))

['fcps', 'g2mg', 'graves', 'h2mg', 'mnist', 'other', 'sipu', 'uci', 'wut']


In [4]:
for ds in iter_benchmark_datasets():
    print(ds.battery, ds.name, ds.X.shape, len(ds.reference_labels))

fcps atom (800, 3) 1
fcps chainlink (1000, 3) 1
fcps engytime (4096, 2) 2
fcps hepta (212, 3) 1
fcps lsun (400, 2) 1
fcps target (770, 2) 2
fcps tetra (400, 3) 1
fcps twodiamonds (800, 2) 1
fcps wingnut (1016, 2) 1
graves dense (200, 2) 1
graves fuzzyx (1000, 2) 5
graves line (250, 2) 1
graves parabolic (1000, 2) 2
graves ring (1000, 2) 1
graves ring_noisy (1050, 2) 1
graves ring_outliers (1030, 2) 2
graves zigzag (250, 2) 2
graves zigzag_noisy (300, 2) 2
graves zigzag_outliers (280, 2) 2
other chameleon_t4_8k (8000, 2) 1
other chameleon_t5_8k (8000, 2) 1
other chameleon_t7_10k (10000, 2) 1
other chameleon_t8_8k (8000, 2) 1
other hdbscan (2309, 2) 1
other iris (150, 4) 1
other iris5 (105, 4) 1
other square (1000, 2) 1
sipu a1 (3000, 2) 1
sipu a2 (5250, 2) 1
sipu a3 (7500, 2) 1
sipu aggregation (788, 2) 1
sipu birch1 (100000, 2) 1
sipu birch2 (100000, 2) 1
sipu compound (399, 2) 5
sipu d31 (3100, 2) 1
sipu flame (240, 2) 2
sipu jain (373, 2) 1
sipu pathbased (300, 2) 2
sipu r15 (600, 2)